Instructions

- All code must be contained in this notebook. No separate code files.
- The code must compile and run without errors.
- Submit as `[your_name].ipynb` with a separate `[your_name]_requirements.txt` file.
- Be prepared to discuss your design decisions in the technical interview.

# Setup (Google Colab)
For running in Colab, implement the T4 for training.

The cell below installs the ONNX packages that
Colab does not have yet (PyTorch, torchvision and NumPy are already installed).

It does nothing on a local machine.

If Colab asks you to restart the session after the install, restart it and run the notebook again from the top.

In [1]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxscript", "onnxruntime"])
    print("Colab detected: onnx, onnxscript and onnxruntime installed")
else:
    print("Not running on Colab: install the packages with pip (see README.md)")

Colab detected: onnx, onnxscript and onnxruntime installed


# Describe the environment that have been used to complete the task
- Python version: 3.13.15 
- GPU used for training (if any): Tesla T4 (CUDA)
- CPU used for inference timing: Intel(R) Xeon(R) CPU @ 2.00GHz (2 logical cores available; inference uses 1 thread)

# Imports, Functions, Global Variables, Classes
Define all shared code in the cell below.

In [ ]:
# ALL THE IMPORTS ARE HERE PRESENTED
# LIBRARIES THAT I USED FOR GENERICAL PURPOSES
# AS MEASURING TIMES IN TRAININGS, BATCHES
# ROUNDING AND OTHER IMPORTANT THINGS
import copy
import math
import os
import platform
import random
import shutil
import sys
import time
from pathlib import Path

# LIBRARIES FOR THE CCNN AND CONVERTION
# TO ONNX
import numpy as np
import onnx
import onnxruntime as ort
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

# LIBRARIES FOR QUANTIZATION
from IPython.display import Markdown, display
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_dynamic,
    quantize_static,
)

# CHARACTERISTICS OF THE CNN, AS WELL AS PLACES
# WHERE THE DATASET IS LOOKED FOR, WHERE THE MODEL
# IS GOING TO BE STORED (I MAKE A FOLDER IF NOT EXISTS)
# I FIXED A SEED TO GUARANTEE REPRODUCIBILITY
SEED = 12
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)


MODEL_NAME = "ccnn"  # NAME OF THE ARCHITECTURE

#ARCHITECTURE GENERATED

CCNN_STEM = 32
# THE FIRST CONVOLUTION IS OF 32
# MEANING THAT I WILL CREATE A
# 32x32x32channels ARRAY BEING A NORMAL
# CONVOLUTION

CCNN_BLOCKS = [ # 5 BLOCKS THAT INTERCALATE BETWEEN STREIDES OF 1 AND 2 TO AVOID ISSUES OF LOSE OF INFORMATION
(64, 1), #32x32x64
(64, 2), #16x16x64
(128, 1), #16x16x128
(128, 2), #8x8x128
(192, 1)] #8x8x192


CCNN_COMPRESS = 128   # HERE IS THE COMPRESSION OF THE FEATURE COMPRESSION OF THE 192 TO 128 CHANNELS

USE_SE = True         # SQUEEZE AND EXCITATION IS USED IN EACH OF THE 5 SEPARABLE BLOCKS
SE_RATIO = 8          # THE SIZE IS 8 BECAUSE A SMALLER RATIO WOULD BE TRANSLATED INTO MORE PARAMETERS

# TRAINING PARAMETERS

EPOCHS = 100
BATCH_SIZE = 128
LR_MAX = 0.1 # WE IMPLEMENT LEARNING RATE MAXIMUM FOR OPTIMIZING
WEIGHT_DECAY = 1e-4  # REGULARIZATION PARAMETER FOR AVOIDING THE OVERFITTING IN THE STEPS
LABEL_SMOOTHING = 0.1 # TO AVOID DISCRETE RESULT PREDICTIONS AND MAKE THEM SOFTER
RETRAIN = False # IN CASE OF EXISTING A TRAINED MODEL DONT RETRAIN IT


# INFERENCE BENCHMARK + QUANTIZATION PART

INF_BATCH = 1   # BATCH USED FOR INFERETION, 1 IMAGE PER BATCH
N_WARMUP = 10
N_RUNS = 50
N_CALIB = 500     # FOR CALIBRATING THE QUANTIZATION, CHECK 500 IMAGES

# WHERE ARE THE MODELS SAVED, FOR ALL THE TASKS

FP32_WEIGHTS = ARTIFACTS / f"{MODEL_NAME}_fp32.pt"
ONNX_FP32 = ARTIFACTS / f"{MODEL_NAME}_fp32.onnx"
ONNX_PREP = ARTIFACTS / f"{MODEL_NAME}_fp32_prep.onnx"
ONNX_INT8_STATIC = ARTIFACTS / f"{MODEL_NAME}_int8_static.onnx"
ONNX_INT8_DYNAMIC = ARTIFACTS / f"{MODEL_NAME}_int8_dynamic.onnx"

# THE MEANS AND STANDARD DEVIATION OF CIFAR DATASET OF EACH RGB CHANNEL

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


RESULTS = {}  # VALUES ARE STORED HERE AS RESULTS

# FUNCTION FOR IMPLEMENTING THE RANDOMNESS OF THE SEED IN ALL CASES

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# FUNCTION FOR ACELERATING THE TRAINING IN THE GPU

torch.backends.cudnn.benchmark = True

# INFORMATION ABOUT THE PYTHON PYTORCH TORCHVISION ONNX ONNXRUNTIME VERSION
# ALSO I IDENTIFY THE USED GPU AND CPU
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__, "| torchvision:", torchvision.__version__)
print("ONNX        :", onnx.__version__, "| ONNXRuntime:", ort.__version__)
print("Train device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only")

def _cpu_name():
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

print("CPU         :", _cpu_name(), "| logical cores:", os.cpu_count())


# LOADING OF THE DATASET AND PREPARATION FOR THE CNN

def load_cifar10():
    # IMAGES ARE LOADED AS THEY ARE
    train_ds = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
    test_ds = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)

    # HERE I TAKE THE MEAN AND STANDARD DEVIATION AND TRANSFORM THEM INTO TENSORS, THE 3 DEFINES THE QUANTITY OF CHANNELS
    mean = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(1, 3, 1, 1)

    # THIS FUNCTION IS VERY IMPORTANT BECAUSE PREPARES THE DATASET FOR OUR COMPACT CONVOLUTIONAL NEURAL NETWORK
    # DS.DATA ARE THE TRAINING IMAGES, MAKES THEM TENSOR, ARE TAKEN TO THE A FLOAT, DEVIDES EVERY PIXEL BY 255 SINCE THE IMAGES ARE RGB,
    # CHANGES THE ORDER OF THE DIMENSIONS AND FINALLY STORES IT CONTIGUOUSLY.
    # NORMALIZES THE IMAGES AND FINALLY PREPARES THE CORRECT LABELS OF EACH
    def prep(ds):
        x = torch.from_numpy(ds.data).float().div_(255.0).permute(0, 3, 1, 2).contiguous()
        x = ((x - mean) / std).contiguous()
        y = torch.tensor(ds.targets, dtype=torch.long)
        return x, y


    x_tr, y_tr = prep(train_ds)
    x_te, y_te = prep(test_ds)
    return x_tr, y_tr, x_te, y_te

# THIS METHOD IS USED FOR DOING DATA AUGMENTATION, A 50% CHANCE HORIZONTAL FLIP, MAKES A RANDOM CROP AND
# AND ADDS A 4 PIXELS PADDING, ALL IN THE GPU DIRECTLY AND AN SPECIFIC BATCH AND FINALLY PUTS IT AGAIN
# INTO THE PYTORCH FORMAT

def augment(x):
    b, dev = x.size(0), x.device
    # FLIP
    flip = torch.rand(b, device=dev) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)

    # PADDING AND RANDOM CROP
    xp = F.pad(x, (4, 4, 4, 4))

    i = torch.randint(0, 9, (b,), device=dev)
    j = torch.randint(0, 9, (b,), device=dev)

    ar = torch.arange(32, device=dev)

    rows = (i[:, None] + ar[None, :])[:, :, None]
    cols = (j[:, None] + ar[None, :])[:, None, :]

    out = xp[torch.arange(b, device=dev)[:, None, None], :, rows, cols]
    return out.permute(0, 3, 1, 2).contiguous()


# DEFINITION OF THE DIFFERENT BLOCKS USED BY THE CCNN
# APPLIES DEPTHWISE CONVOLUTION FIRST, TAKINIG IMPORTANT INFORMATION FOR EACH CHANNEL SEPARATEDLY,
# THEN POINTWISE CONVOLUTION LATER TAKES ALL THE POINTS OF EACH CHANNEL AND CREATES MORE CHANNELS FROM IT
# BATCH NORMALIZATION NORMALIZES ACTIVATION FUNCTION
# APPLICATION OF A RELU IS IMPORTANT FOR GIVING NO LINEARITY

class SeparableBlock(nn.Module):

    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(cin, cin, 3, stride, 1, groups=cin, bias=False) #32x32x32 is now 32x32x32 AND SO ON FOR ALL THE 5 LAYERS
        self.bn1 = nn.BatchNorm2d(cin)
        self.pointwise = nn.Conv2d(cin, cout, 1, bias=False) #32x32x32 is now 32x32x64 AND SO ON FOR ALL THE 5 LAYERS
        self.bn2 = nn.BatchNorm2d(cout)


    def forward(self, x):
        x = torch.nn.functional.relu(self.bn1(self.depthwise(x)))
        return F.relu(self.bn2(self.pointwise(x)))


# DEFINITION OF THE SQUEEZE AND EXCITATION BLOCK, WHICH TAKES THE END OF EACH CONVOLUTIVE BLOCK,
# COMPRESSES THE INFORMATION BASED ON AN SPECIFIC RATIO (IN OUR CASE 8) AND EXPANDS IT AFTERWARDS
# LEARNING THE IMPORTANCE OF EACH CHANNEL, FIRST APPLYING A GLOBAL AVERAGE POOL WITHIN THE BLOCK

class SEBlock(nn.Module):
    def __init__(self, channels, ratio=SE_RATIO):
        super().__init__()
        mid = max(channels // ratio, 4)
        self.fc1 = nn.Conv2d(channels, mid, 1)
        self.fc2 = nn.Conv2d(mid, channels, 1)


    def forward(self, x):
        s = torch.nn.functional.adaptive_avg_pool2d(x, 1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(s))))

        return x * s

# DESCRIPTION OF THE ARCHITECTURE, IN PARTICULAR CONSISTS ON FIRST A 3X3 NORMAL CONVOLUTION STEM
# THEN 5 BLOCKS CONSISTING ON ONE SEPARABLE BLOCK WITH AN SQUEEZE AND EXCITATE BLOCK TO RECALIBRATE
# THE IMPORTANCE OF EACH CHANNEL, AFTER THIS HAVING A 8X8X192 PER IMAGE.
# IT IS APPLIED A FEATURE COMPRESSION (WITH 1X1 CONV) OF 128.
# GLOBAL AVERAGE IS APPLIED ONCE MORE TO REDUCE FROM 8X8X128 TO 1X1X128,
# FINALLY A FLATTEN IS USED TO CREATE A VECTOR OF 128 VALUES AND A LINEAR LAYER THAT
# CHANGES 128 INTO A 10 LOGITS

class CCNN(nn.Module):
    def __init__(self, stem=CCNN_STEM, blocks=CCNN_BLOCKS, compress=CCNN_COMPRESS,
                 use_se=USE_SE, se_ratio=SE_RATIO, num_classes=10):

        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, stem, 3, 1, 1, bias=False), nn.BatchNorm2d(stem), nn.ReLU())
        layers, cin = [], stem
        for cout, stride in blocks:
            layers.append(SeparableBlock(cin, cout, stride))
            if use_se:
                layers.append(SEBlock(cout, se_ratio))
            cin = cout

        self.features = nn.Sequential(*layers)

        self.compress = nn.Sequential(nn.Conv2d(cin, compress, 1, bias=False), nn.BatchNorm2d(compress), nn.ReLU())

        self.fc = nn.Linear(compress, num_classes)


    def forward(self, x):
        x = self.compress(self.features(self.stem(x)))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))


# CREATING AN INSTANCE OF A CCNN

def build_model():
    return CCNN()

# COUNTS THE TOTAL NUMBER OF TRAINABLE PARAMETERS

def count_params(model):
    return sum(p.numel() for p in model.parameters())

# ESTIMATION OF THE MEMORY FOOTPRINT

def model_size_kb(model):
    n_bytes = sum(t.numel() * t.element_size() for t in model.state_dict().values() if t.is_floating_point())
    return n_bytes / 1024

# RETURNS THE ACTUAL FILE SIZE IN KiB

def file_kb(path):
    return Path(path).stat().st_size / 1024


# TRAINING AND EVALUATION OF THE MODEL

def train_fp32(model, x_tr, y_tr, x_val, y_val, epochs=EPOCHS):
    model.to(DEVICE)
    xg, yg = x_tr.to(DEVICE), y_tr.to(DEVICE)
    steps_per_epoch = math.ceil(len(xg) / BATCH_SIZE)

    # IT IS USED A STOCHASTIC GRADIENT DESCENT TO REDUCE THE ERROR, MOMENTUM TO SOFTEN OSCILATIONS
    # WITH NESTEROV IS TRIED TO UPGRADE THE TRAYECTORY AND A PENALIZATION (Weight_decay) TO REGULARIZE
    # THE MODEL AND AVOID OVERFITTING

    opt = torch.optim.SGD(model.parameters(), lr=LR_MAX, momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)

    # AUTIOMATIC ADJUSTMENT OF THE LEARNING RATE DURING TRAINING

    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR_MAX, epochs=epochs, steps_per_epoch=steps_per_epoch, pct_start=0.15)

    # THE LOSS FUNCTION IS CROSSENTROPY, SINCE CIFAR10 IS A MULTICLASS PROBLEM WITH 10 EXCLUSIVE CLASSES

    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(xg), device=DEVICE)
        running = 0.0

        for i in range(0, len(xg), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb, yb = augment(xg[idx]), yg[idx]
            loss = crit(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            running += loss.item() * len(idx)

        if epoch % 10 == 0 or epoch == 1 or epoch == epochs:
            val_acc = evaluate_torch(model, x_val, y_val)
            print(f"epoch {epoch:3d}/{epochs} | train loss {running / len(xg):.4f} | val acc {val_acc:.2f}%")

    return model

# STOP OF GRADIENT CALCULATION AND AVOID EVALUATION

@torch.inference_mode()

# FUNCTION THAT EVALUATES THE PERFORMANCE

def evaluate_torch(model, x, y, bs=1000, device=None):
    device = device or next(model.parameters()).device
    model.eval()
    correct = 0

    for i in range(0, len(x), bs):
        correct += (model(x[i:i + bs].to(device)).argmax(1).cpu() == y[i:i + bs]).sum().item()
    return 100.0 * correct / len(x)


# ONNX EXPORT HELP

# USES THE ALREADY TRAINED MODEL, DEFINES ITS PATH AND DEFINES THE OPERATIONS OF ONNX THAT WILL BE USED
# M IS A COPY OF THE MODEL RAN IN THE CPU AND ACTIVATES EVALUATION MODE
# GENERATES A RANDOM IMAGE THAT IS THE VALUES THAT WILL BE USED BY THE EXPORT TO RUN THE MODEL
# CONFIGURES INPUT AND OUTPUT NAMES AND ALLOWS A DYNAMIC BATCH SIZE

def export_onnx(model, path, opset=17):
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    kwargs = dict(
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={"input": {0: "batch"},
                      "logits": {0: "batch"}},
        opset_version=opset, do_constant_folding=True,
    )

    try:
        torch.onnx.export(m, dummy, str(path), dynamo=False, **kwargs)

    except Exception as exc:  # IN CASE OF FAILURE, GIVES ANOTHER SHAPE OF EXPORT
        print("legacy exporter failed (", type(exc).__name__, ") now we are using default exporter")
        torch.onnx.export(m, dummy, str(path), **kwargs)
    onnx.checker.check_model(onnx.load(str(path)))


# RUNNING OF THE EXPORTED FILE ONNX IN ONNX RUNTIME
# ASSUMES ONLY 1 THREAD AS IT IS IMPLICIT ON THE TASK

def make_session(path, threads=1):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])

# OBTAIN PREDICTIONS ONNX AND RUNS THE MODEL

def logits_ort(sess, x_np, bs=500):
    name = sess.get_inputs()[0].name
    return np.concatenate([sess.run(None, {name: x_np[i:i + bs]})[0] for i in range(0, len(x_np), bs)])

# ACCURACY IS CALCULATED TO ONNX FILE WITH 500 BATCHES

def evaluate_ort(sess, x_np, y_np, bs=500):
    return 100.0 * float((logits_ort(sess, x_np, bs).argmax(1) == y_np).mean())


# INFERENCE TIME IS MEASURED TAKING AWAY THE WARM UP NUMBER
# THEN THE EXECUTIONS ARE MEASURED N_RUNS NUMBER, THEN CALCULATES
# THE MEDIAN OF THE RECORDED TIMES

def benchmark(fn, n_warmup=N_WARMUP, n_runs=N_RUNS):
    for _ in range(n_warmup):
        fn()

    times = []

    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000.0)

    return float(np.median(times))

# CLASS CREATED FOR STATIC QUANTIZATION
# THE IMAGES ARE X_NP (IN NUMPY) AND 50 IMAGES PER BATCH
# WE INITALIZE THE BATCHES CREATING A DICTIONARY OF EACH
# WE CREATE AN ITERATOR THAT GOES POSITION BY POSITION

class CifarCalibrationReader(CalibrationDataReader):
    def __init__(self, x_np, input_name, bs=50):
        self.batches = [{input_name: np.ascontiguousarray(x_np[i:i + bs])} for i in range(0, len(x_np), bs)]
        self.it = iter(self.batches)


    def get_next(self):
        return next(self.it, None)


    def rewind(self):
        self.it = iter(self.batches)

Python      : 3.13.15
PyTorch     : 2.11.0+cu130 | torchvision: 0.26.0+cu130
ONNX        : 1.23.2 | ONNXRuntime: 1.31.0
Train device: cuda | Tesla T4
CPU         : Intel(R) Xeon(R) CPU @ 2.00GHz | logical cores: 2


# 2.1 Design of a Compact CNN

**Requirements:**
- Model size: < 500 KB (FP32)
- Target test accuracy: ≥ 85%

In [3]:
# IMPLEMENTATION OF THE TRAINING, WE LOAD THE DATASET TRAINING VALIDATION AND TESTING, LOAD_CIFAR10()
# ALREADY HAS PREPARED EVERYTHING, IN TOTAL 60K IMAGES.
# SHUFFEL OF IMAGES IS PRODUCED AND WE SPLIT TRAINING AND VALIDATION 45k 5k
# WE SELECT IMAGES AND ITS LABELS

x_all, y_all, x_test, y_test = load_cifar10()
perm = torch.randperm(len(x_all), generator=torch.Generator().manual_seed(SEED))
val_idx, train_idx = perm[:5000], perm[5000:]
x_train, y_train = x_all[train_idx], y_all[train_idx]
x_val, y_val = x_all[val_idx], y_all[val_idx]
print("train/val/test:", len(x_train), len(x_val), len(x_test))

# IMPLEMENTATION OF THE CCNN
# COUNTING OF PARAMETERS AND CALCULATION OF SIZE, IF DOESNT FIT, THEN ALERT ARAISES

model = build_model()
n_params = count_params(model)
size_kb = model_size_kb(model)
print(f"model: {MODEL_NAME} | parameters: {n_params:,} | FP32 footprint (params + BN stats): {size_kb:.1f} KB")
assert size_kb < 500, "model too large for the 500 KB budget"

# TRAINING EXPECTED IF THERE IS NO MODEL ALREADY, OTHERWISE SKIPS THIS OTHERWISE LOAD

if FP32_WEIGHTS.exists() and not RETRAIN:
    model.load_state_dict(torch.load(FP32_WEIGHTS, map_location="cpu"))
    print("loaded weights from", FP32_WEIGHTS)
else:
    set_seed()
    t0 = time.time()
    train_fp32(model, x_train, y_train, x_val, y_val)
    print(f"training time: {(time.time() - t0) / 60:.1f} min")
    torch.save(model.state_dict(), FP32_WEIGHTS)

# CALCULATION OF VALIDATION AND TEST ACCURACY AND SAVED IT AS A .PT IT ALSO SAVES
# THE CHARACTERISTICS OF THE PYTORCH CNN FOR LATER IN RESULTS

model.to(DEVICE)
val_acc_fp32 = evaluate_torch(model, x_val, y_val)
test_acc_fp32 = evaluate_torch(model, x_test, y_test)
print(f"FP32 PyTorch | val acc {val_acc_fp32:.2f}% | TEST acc {test_acc_fp32:.2f}%")
print(f"saved .pt file: {file_kb(FP32_WEIGHTS):.1f} KB")

RESULTS["fp32_torch"] = {"size_kb": size_kb, "acc": test_acc_fp32}

100%|██████████| 170M/170M [53:54<00:00, 52.7kB/s]


train/val/test: 45000 5000 10000
model: ccnn | parameters: 108,178 | FP32 footprint (params + BN stats): 431.6 KB
epoch   1/100 | train loss 1.9414 | val acc 41.06%
epoch  10/100 | train loss 0.9805 | val acc 78.78%
epoch  20/100 | train loss 0.8368 | val acc 82.56%
epoch  30/100 | train loss 0.7800 | val acc 84.10%
epoch  40/100 | train loss 0.7489 | val acc 84.88%
epoch  50/100 | train loss 0.7232 | val acc 85.80%
epoch  60/100 | train loss 0.6966 | val acc 86.42%
epoch  70/100 | train loss 0.6726 | val acc 88.34%
epoch  80/100 | train loss 0.6299 | val acc 89.14%
epoch  90/100 | train loss 0.5979 | val acc 90.38%
epoch 100/100 | train loss 0.5852 | val acc 90.48%
training time: 11.9 min
FP32 PyTorch | val acc 90.48% | TEST acc 89.88%
saved .pt file: 462.5 KB


**Fill-in the Results:**
- Model Size: 431.6 KB
- Test Accuracy: 89.88%

**Provide brief notes (architecture choice, training decisions):**
- ### Architecture Choice
The Compact Convolutional Neural Network was designed to balance classification accuracy, model size and computational efficiency for resource-constrained devices.

The architecture begins with a standard 3×3 convolutional stem with 32 output channels, followed by five depthwise separable convolution blocks, to reduce the number of parameters and computational operations compared to standard convolutions.

Also, each block includes a Squeeze-and-Excitation module with a reduction ratio of 8 to recalibrate channel importance while limiting additional parameters, with strides of 1 and 2 used to progressively reduce spatial dimensions.

A final 1×1 convolution reduces the feature channels from 192 to 128, followed by Global Average Pooling and a linear classifier for the 10 CIFAR-10 classes.

**The resulting architecture contains **108,178 trainable parameters**, with the goal of satisfying the 500 KB FP32 ONNX model-size constraint.**

- ### Training Decisions

The CIFAR-10 dataset was divided into 45,000 training images, 5,000 validation images and 10,000 official test images.

Training was configured for 100 epochs as a practical balance between model convergence and computational cost.

The batch size of 128 to balance the GPU consumption with training efficiency.

The optimizer was using stochastic gradient descent with momentum of 0.9 to avoid oscilations, Nesterov acceleration to improve the direction of the optimization steps and support faster convergence, and weight decay (1e-4) for regularization.

A OneCycle learning rate scheduler was used with a maximum learning rate of 0.1 in the first 15% of the training.

Cross-entropy loss was selected because CIFAR-10 is a multi-class classification problem with 10 mutually exclusive classes. Label smoothing of 0.1 to avoid false confidence applied in the multi-class classification.

Data augmentation included random horizontal flipping, padding and random cropping to improve generalization.

# 2.2 Inference using ONNXRuntime (CPU)

Export your model to ONNX and run inference using ONNXRuntime (CPU).

In [4]:
# CONVERSION TO ONNX FP32 AND CHECKS AGAIN THE SIZE, TO COMPARE WITH PYTORCH

export_onnx(model, ONNX_FP32)
onnx_kb = file_kb(ONNX_FP32)
print(f"ONNX FP32 file: {onnx_kb:.1f} KB")

# THE DATA IS CONVERTED TO NUMPY SINCE PYTORCH USES TENSORS AND ONNX ARRAYS OF NUMPY

x_test_np, y_test_np = x_test.numpy(), y_test.numpy()
x_val_np, y_val_np = x_val.numpy(), y_val.numpy()

# CREATES AN ONNX RUNTIME SESSION AND EVALUATES THE TEST ACCURACY
# TO CHECK IF THE CONVERSION FROM PYTORCH TO ONNX PRESERVED PERFORMANCE

sess_fp32 = make_session(ONNX_FP32)
acc_onnx = evaluate_ort(sess_fp32, x_test_np, y_test_np)
print(f"ONNXRuntime FP32 | TEST acc {acc_onnx:.2f}%  (PyTorch: {test_acc_fp32:.2f}%)")

# COMPARES THE LOGITS PRODUCED BY PYTORCH AND ONNX RUNTIME
# TO VERIFY THAT BOTH MODELS PRODUCE NEARLY IDENTICAL OUTPUTS
# AND PREDICT THE SAME CLASSES

model_cpu = copy.deepcopy(model).cpu().eval()
with torch.inference_mode():
    logits_t = torch.cat([model_cpu(x_test[i:i + 1000]) for i in range(0, len(x_test), 1000)]).numpy()
logits_o = logits_ort(sess_fp32, x_test_np)
diff = np.abs(logits_t - logits_o)

agree = 100.0 * float((logits_t.argmax(1) == logits_o.argmax(1)).mean())

print(f"logits | max |diff| {diff.max():.2e} | mean |diff| {diff.mean():.2e} | same top-1 class: {agree:.2f}%")

# MEASURES THE INFERENCE TIME OF PYTORCH AND ONNX RUNTIME
# USING ONE CPU THREAD AND A BATCH SIZE OF ONE
# COMPARES THEIR EXECUTION SPEED

torch.set_num_threads(1)
xb_t, xb_np = x_test[:INF_BATCH], x_test_np[:INF_BATCH]
in_name = sess_fp32.get_inputs()[0].name


@torch.inference_mode()
def run_torch():
    model_cpu(xb_t)

# FINALLY BENCHMARKS PYTORCH AND ONNX RUNTIME INFERENCE ON THE CPU, AS WELL AS
# MEASURING THE EXECUTION TIME USING A SINGLE THREAD AND BATCH SIZE OF ONE AND
# CALCULATES THE SPEED-UP OF ONNX RUNTIME COMPARED TO PYTORCH

ms_torch = benchmark(run_torch)
ms_onnx = benchmark(lambda: sess_fp32.run(None, {in_name: xb_np}))
print(f"Latency (batch={INF_BATCH}, 1 CPU thread): PyTorch {ms_torch:.2f} ms | ONNXRuntime {ms_onnx:.2f} ms "
      f"| speed-up x{ms_torch / ms_onnx:.2f}")

RESULTS["fp32_torch"]["ms"] = ms_torch
RESULTS["fp32_onnx"] = {"size_kb": onnx_kb, "acc": acc_onnx, "ms": ms_onnx}

/tmp/ipykernel_552/1088347608.py:359: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(m, dummy, str(path), dynamo=False, **kwargs)


ONNX FP32 file: 429.2 KB
ONNXRuntime FP32 | TEST acc 89.88%  (PyTorch: 89.88%)
logits | max |diff| 1.41e-05 | mean |diff| 8.24e-07 | same top-1 class: 100.00%
Latency (batch=1, 1 CPU thread): PyTorch 3.82 ms | ONNXRuntime 0.63 ms | speed-up x6.05


**Fill-in the Results**

- ONNX Model Size: 429.2 KB
- Test Accuracy (ONNX): 89.88%
- Inference Time (FP32 Original): 3.82 ms/batch
- Inference Time (ONNX FP32): 0.63 ms/batch

**Provide brief comparison/analysis:**

The ONNX FP32 model achieved the same test accuracy as the original PyTorch model 89.88% with 100% agreement in predicted classes with the comparison of the logits.

The exported ONNX model 429.2 KB was smaller than the PyTorch .pt file 462.5 KB. This difference can be explained by ONNX graph optimizations, such as constant folding and potential BatchNorm fusion, as well as differences in file serialization and metadata.

ONNX Runtime achieved 0.63 ms/batch, compared with 3.82 ms/batch for PyTorch, resulting in a 6.05× speed-up. This improvement may be attributed to graph optimizations and lower Python execution overhead.

However, these measurements were performed on a CPU using a batch size of 1 and a single inference thread. They represent a CPU performance comparison rather than actual inference latency on the target microcontroller.

# 2.3 Post Training Quantization (Static)
Perform INT8 static quantization. Target: < 5% accuracy drop from FP32.

In [5]:
# PREPROCESSING OF THE ONNX FP32 BEFORE QUANTIZATION TO REACH A BETTER OPTIMIZATION
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    quant_pre_process(str(ONNX_FP32), str(ONNX_PREP))
    quant_model_input = ONNX_PREP
except Exception as exc:
    print("quant_pre_process skipped:", type(exc).__name__, exc)
    quant_model_input = ONNX_FP32
input_name = make_session(quant_model_input).get_inputs()[0].name

# WITH THE SEED FIXED, WE CALIBRATE IN OUR EXAMPLE 500 IMAGES FROM THE TRAINING
rng = np.random.default_rng(SEED)
calib_idx = torch.from_numpy(rng.choice(len(x_train), N_CALIB, replace=False))
x_calib_np = x_train[calib_idx].numpy()

# WE USE DIFFERENT 5 STATIC CONFIGURATIONS OF INT8, USED ON THE VALIDATION DATA OF 5k IMAGES
STATIC_CONFIGS = [
    dict(name="minmax_u8s8", method=CalibrationMethod.MinMax, act=QuantType.QUInt8, reduce_range=False),
    dict(name="minmax_u8s8_reduce_range", method=CalibrationMethod.MinMax, act=QuantType.QUInt8, reduce_range=True),
    dict(name="percentile_u8s8", method=CalibrationMethod.Percentile, act=QuantType.QUInt8, reduce_range=False),
    dict(name="entropy_u8s8", method=CalibrationMethod.Entropy, act=QuantType.QUInt8, reduce_range=False),
    dict(name="minmax_s8s8", method=CalibrationMethod.MinMax, act=QuantType.QInt8, reduce_range=False),
]

# EVALUATES EACH CONFIGURATION ON THE VALIDATION SET.
# SELECTS THE CONFIGURATION WITH THE HIGHEST VALIDATION ACCURACY

val_ref = evaluate_ort(sess_fp32, x_val_np, y_val_np)
print(f"reference FP32 ONNX val acc: {val_ref:.2f}%")
sweep = []

# QUANTIZES THE FP32 ONNX MODEL USING EACH CONFIGURATION

for cfg in STATIC_CONFIGS:
    out_path = ARTIFACTS / f"{MODEL_NAME}_int8_static_{cfg['name']}.onnx"
    try:
        quantize_static(
            str(quant_model_input),
            str(out_path),
            CifarCalibrationReader(x_calib_np, input_name),
            quant_format=QuantFormat.QDQ,
            per_channel=True,
            reduce_range=cfg["reduce_range"],
            activation_type=cfg["act"],
            weight_type=QuantType.QInt8,
            calibrate_method=cfg["method"],
        )

        # EVALUATION OF EACH MODEL WITH VALIDATION

        acc_v = evaluate_ort(make_session(out_path), x_val_np, y_val_np)
        sweep.append((cfg["name"], acc_v, out_path))
        print(f"{cfg['name']:28s} val acc {acc_v:.2f}%  size {file_kb(out_path):.1f} KB")
    except Exception as exc:
        print(f"{cfg['name']:28s} FAILED: {type(exc).__name__}: {exc}")

if not sweep:
    raise RuntimeError("no static quantization configuration worked")

# BEST MODEL OF INT8

best_name, best_val, best_path = max(sweep, key=lambda t: t[1])
shutil.copyfile(best_path, ONNX_INT8_STATIC)
print("the best calibration is reached with:", best_name)

# FINAL EVALUATION OF THE SELECTED STATIC INT8 MODEL, MEASURING TEST ACCURACY,
# FILE SIZE AND CPU INFERENCE LATENCY, WE AIM TO COMPARE THE ACCURACY DROP AND
# LATENCY AGAINST THE FP32 BASELINE, USES ONE CPU THREAD AND A BATCH SIZE OF ONE

sess_int8 = make_session(ONNX_INT8_STATIC)
acc_int8 = evaluate_ort(sess_int8, x_test_np, y_test_np)
ms_int8 = benchmark(lambda: sess_int8.run(None, {in_name: xb_np}))
size_int8 = file_kb(ONNX_INT8_STATIC)
drop = test_acc_fp32 - acc_int8
print(f"INT8 static | size {size_int8:.1f} KB | TEST acc {acc_int8:.2f}% | drop vs FP32 {drop:.2f} pp "
      f"| {ms_int8:.2f} ms/batch (FP32 ONNX: {ms_onnx:.2f})")
print("accuracy-drop requirement (< 5 pp):", "OK" if drop < 5 else "NOT MET")

RESULTS["int8_static"] = {"size_kb": size_int8, "acc": acc_int8, "ms": ms_int8, "config": best_name}

reference FP32 ONNX val acc: 90.48%
minmax_u8s8                  val acc 87.96%  size 186.3 KB
minmax_u8s8_reduce_range     val acc 90.22%  size 186.3 KB
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 58
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
percentile_u8s8              val acc 88.30%  size 186.3 KB
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 58
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
entropy_u8s8                 val acc 87.96%  size 186.3 KB
minmax_s8s8                  val acc 87.96%  size 186.6 KB
the best calibration is reached with: minmax_u8s8_reduce_range
INT8 static | size 186.3 KB | TEST acc 89.58% | drop vs FP32 0.30 pp | 0.50 ms/batch (FP32 ONNX: 0.63)
accuracy-drop requirement (< 5 pp): OK


### Fill-in the Results

- **INT8 Model Size:** 186.3 KB
- **INT8 Test Accuracy:** 89.58%
- **Accuracy Drop:** 0.30 percentage points
- **Inference Time (INT8):** 0.50 ms/batch

### Quantization Settings Used

- **Format:** QDQ (QuantizeLinear/DequantizeLinear), with per-channel INT8 weights and UINT8 activations.
- **Calibration:** 500 randomly selected training images without augmentation. MinMax, Percentile, Entropy and reduced-range configurations were compared using 5,000 validation images.
- **Selected Configuration:** `minmax_u8s8_reduce_range`, achieving the highest validation accuracy of 90.22%.
- **Pre-processing:** `quant_pre_process` was applied before quantization for shape inference and graph preprocessing.

### Brief Comparison / Analysis

Static INT8 quantization reduced the ONNX model size from 429.2 KB to 186.3 KB, making it approximately **2.3x smaller**, corresponding to a **56.6% reduction**.

The INT8 model achieved 89.58% test accuracy, compared with 89.88% for FP32, resulting in a loss of only **0.30 percentage points**.

CPU inference latency improved from 0.63 ms/batch in the FP32 ONNX to 0.50 ms/batch in INT8, representing approximately a **1.26× speed-up**.

The relatively modest speed improvement may be explained by the small size of the CNN, quantization/dequantization overhead and the CPU's available integer-operation optimizations. The main advantage is the significant reduction in model storage requirements while preserving classification accuracy.

These results were measured using ONNX Runtime on a CPU with batch size 1 and a single inference thread rather than on the target microcontroller.

# **OPTIONAL - BONUS** 2.4 Post Training Quantization (Dynamic)

*(Optional)* Perform INT8 dynamic quantization.

In [8]:

# DYNAMIC QUANTIZATION CALCULATES ACTIVATION QUANTIZATION PARAMETERS
# DURING INFERENCE FOR COMPATIBLE OPERATIONS, WITHOUT PRIOR CALIBRATION.
# WE TEST TWO CONFIGURATIONS TO COMPARE THEIR PERFORMANCE.

DYNAMIC_CONFIGS = [
    # ATTEMPTS TO QUANTIZE CONVOLUTIONAL AND LINEAR OPERATIONS.
    dict(
        name="all_ops_conv_gemm",
        ops=["Conv", "Gemm", "MatMul"],
        wtype=QuantType.QUInt8
    ),

    # QUANTIZES ONLY COMPATIBLE LINEAR OPERATIONS.
    # USED AS A BASELINE, SINCE MOST OF OUR CNN IS CONVOLUTIONAL.
    dict(
        name="linear_only",
        ops=["Gemm", "MatMul"],
        wtype=QuantType.QInt8
    ),
]


# BOTH CONFIGURATIONS ARE QUANTIZED AND EVALUATED SEPARATELY.
# NO CALIBRATION DATA IS REQUIRED FOR DYNAMIC QUANTIZATION.
# VALIDATION AND TEST ACCURACY, MODEL SIZE AND LATENCY ARE RECORDED.

dyn_results = []

for cfg in DYNAMIC_CONFIGS:

    out_path = ARTIFACTS / f"{MODEL_NAME}_int8_dynamic_{cfg['name']}.onnx"

    try:
        # GENERATE THE DYNAMICALLY QUANTIZED ONNX MODEL.
        quantize_dynamic(
            str(quant_model_input),
            str(out_path),
            op_types_to_quantize=cfg["ops"],
            weight_type=cfg["wtype"]
        )

        # CREATE AN ONNX RUNTIME SESSION.
        sess_dyn = make_session(out_path)

        # EVALUATE VALIDATION ACCURACY.
        acc_val = evaluate_ort(sess_dyn, x_val_np, y_val_np)

        # EVALUATE TEST ACCURACY.
        acc_test = evaluate_ort(sess_dyn, x_test_np, y_test_np)

        # MEASURE CPU INFERENCE LATENCY AND MODEL FILE SIZE.
        input_name = sess_dyn.get_inputs()[0].name

        ms_dyn = benchmark(
            lambda: sess_dyn.run(None, {input_name: xb_np})
        )

        size_dyn = file_kb(out_path)

        # CALCULATE ACCURACY DROP RELATIVE TO FP32.
        drop = test_acc_fp32 - acc_test

        # STORE ALL RESULTS FOR THIS CONFIGURATION.
        dyn_results.append({
            "config": cfg["name"],
            "val_acc": acc_val,
            "test_acc": acc_test,
            "size_kb": size_dyn,
            "ms": ms_dyn,
            "drop_pp": drop,
            "path": out_path
        })

        RESULTS[f"int8_dynamic_{cfg['name']}"] = {
            "size_kb": size_dyn,
            "acc": acc_test,
            "ms": ms_dyn,
            "config": cfg["name"]
        }

        print(
            f"\nDynamic INT8 ({cfg['name']})"
            f"\nValidation accuracy: {acc_val:.2f}%"
            f"\nTest accuracy:       {acc_test:.2f}%"
            f"\nAccuracy drop:       {drop:.2f} pp"
            f"\nModel size:          {size_dyn:.1f} KB"
            f"\nInference latency:   {ms_dyn:.2f} ms/batch"
        )

    except Exception as exc:
        print(
            f"{cfg['name']} FAILED: "
            f"{type(exc).__name__}: {str(exc)[:200]}"
        )


# DISPLAY A FINAL COMPARISON BETWEEN BOTH CONFIGURATIONS.

if dyn_results:
    print("\nDYNAMIC QUANTIZATION COMPARISON")

    for result in dyn_results:
        print(
            f"{result['config']:20s} | "
            f"VAL {result['val_acc']:.2f}% | "
            f"TEST {result['test_acc']:.2f}% | "
            f"DROP {result['drop_pp']:.2f} pp | "
            f"SIZE {result['size_kb']:.1f} KB | "
            f"LATENCY {result['ms']:.2f} ms"
        )
else:
    print(
        "Dynamic quantization did not produce "
        "a runnable model on this ONNXRuntime build"
    )



Dynamic INT8 (all_ops_conv_gemm)
Validation accuracy: 90.28%
Test accuracy:       89.78%
Accuracy drop:       0.10 pp
Model size:          159.1 KB
Inference latency:   1.85 ms/batch

Dynamic INT8 (linear_only)
Validation accuracy: 90.50%
Test accuracy:       89.90%
Accuracy drop:       -0.02 pp
Model size:          430.5 KB
Inference latency:   0.42 ms/batch

DYNAMIC QUANTIZATION COMPARISON
all_ops_conv_gemm    | VAL 90.28% | TEST 89.78% | DROP 0.10 pp | SIZE 159.1 KB | LATENCY 1.85 ms
linear_only          | VAL 90.50% | TEST 89.90% | DROP -0.02 pp | SIZE 430.5 KB | LATENCY 0.42 ms


**Results:**
- INT8 Model Size: 159.1 KB
- INT8 Test Accuracy: 89.78%
- Accuracy Drop: 0.1 percentage points 
- Inference Time INT8: 1.85 ms/batch

**Comparison with static quantization:**

- `all_ops_conv_gemm` produced the smallest model (159.1 KB) with 89.78% accuracy, but was considerably slower (1.85 ms).

- `linear_only` achieved the highest accuracy (89.90%) and lowest latency (0.42 ms), but quantizes only a small part of the CNN, leaving most layers in FP32 and providing almost no model-size reduction (430.5 KB).

- **Static INT8 offers the best overall balance** for this TinyML project, achieving 186.3 KB, a 89.58% accuracy in the testing and 0.50 ms inference latency.


# Summary Table

| Metric | FP32 (Original) | FP32 (ONNX) | INT8 Static | INT8 Dynamic (Conv + Gemm) | INT8 Dynamic (Linear Only) |
|---|---:|---:|---:|---:|---:|
| Size (KB) | 431.6 | 429.2 | 186.3 | 159.1 | 430.5 |
| Accuracy (%) | 89.88 | 89.88 | 89.58 | 89.78 | 89.90 |
| Inference (ms/batch, 1 CPU thread) | 3.82 | 0.63 | 0.50 | 1.85 | 0.42 |



In [13]:

# FINAL COMPARISON OF FP32, STATIC INT8 AND BOTH DYNAMIC INT8 CONFIGURATIONS.
def _fmt(key, field, nd=2):
    v = RESULTS.get(key, {}).get(field)
    return "-" if v is None else f"{v:.{nd}f}"


keys = [
    "fp32_torch",
    "fp32_onnx",
    "int8_static",
    "int8_dynamic_all_ops_conv_gemm",
    "int8_dynamic_linear_only"
]

table = (
    "| Metric | FP32 (Original) | FP32 (ONNX) | INT8 Static | INT8 Dynamic (Conv + Gemm) | INT8 Dynamic (Linear Only) |\n"
    "|--------|-----------------|-------------|-------------|---------------------------|---------------------------|\n"
    f"| Size (KB) | {' | '.join(_fmt(k, 'size_kb', 1) for k in keys)} |\n"
    f"| Accuracy (%) | {' | '.join(_fmt(k, 'acc') for k in keys)} |\n"
    f"| Inference (ms/batch of {INF_BATCH}, 1 CPU thread) | {' | '.join(_fmt(k, 'ms') for k in keys)} |\n"
)

print(table)
display(Markdown(table))


| Metric | FP32 (Original) | FP32 (ONNX) | INT8 Static | INT8 Dynamic (Conv + Gemm) | INT8 Dynamic (Linear Only) |
|--------|-----------------|-------------|-------------|---------------------------|---------------------------|
| Size (KB) | 431.6 | 429.2 | 186.3 | 159.1 | 430.5 |
| Accuracy (%) | 89.88 | 89.88 | 89.58 | 89.78 | 89.90 |
| Inference (ms/batch of 1, 1 CPU thread) | 3.82 | 0.63 | 0.50 | 1.85 | 0.42 |



| Metric | FP32 (Original) | FP32 (ONNX) | INT8 Static | INT8 Dynamic (Conv + Gemm) | INT8 Dynamic (Linear Only) |
|--------|-----------------|-------------|-------------|---------------------------|---------------------------|
| Size (KB) | 431.6 | 429.2 | 186.3 | 159.1 | 430.5 |
| Accuracy (%) | 89.88 | 89.88 | 89.58 | 89.78 | 89.90 |
| Inference (ms/batch of 1, 1 CPU thread) | 3.82 | 0.63 | 0.50 | 1.85 | 0.42 |


In [ ]:
# COMPRESS THE TRAINED MODELS INTO A ZIP FILE
# AND DOWNLOAD THEM DIRECTLY TO THE COMPUTER.

import shutil

zip_path = shutil.make_archive(
    "artifacts",
    "zip",
    root_dir=ARTIFACTS
)

print("Models saved in:", ARTIFACTS.resolve())
print("ZIP archive created:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

Models saved in: /content/artifacts
ZIP archive created: /content/artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>